# Sample Filter Diagnostics

Quantify the impact of each screening filter on the analysis sample before applying them in `prepare_data.py`.

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../data")

daily = pd.read_csv(DATA / "analysis_daily.csv", parse_dates=["Date"])
fund  = pd.read_csv(DATA / "idx_fundamentals.csv", parse_dates=["Date", "IPO_Date"])
# Merge sector + BVPS + IPO_Date onto daily panel via most recent fundamentals
fund["YearMonth"] = fund["Date"].dt.to_period("M").astype(str)

fund[fund["BVPS"].notna()].head(5)

,Instrument,Date,Market_Cap,ROA,IPO_Date,DPS_Raw,BVPS,Sector,Shares_Outstanding,Market_Cap_Bil,Div_Payer,Months_Since_Listing,YearMonth
0,AADI.JK,2024-12-30,6.599391e+13,NaN,1997-12-09,NaN,0.560686,Consumer Non-Cyclicals,7.786892e+09,65993.907666,0,325,2024-12
1,AADI.JK,2025-01-31,7.358613e+13,NaN,1997-12-09,NaN,0.560686,Consumer Non-Cyclicals,7.786892e+09,73586.127132,0,326,2025-01
2,AADI.JK,2025-02-28,5.236685e+13,NaN,1997-12-09,NaN,0.560686,Consumer Non-Cyclicals,7.786892e+09,52366.847086,0,327,2025-02
3,AADI.JK,2025-03-27,5.080947e+13,NaN,1997-12-09,0.05425,0.385120,Consumer Non-Cyclicals,7.786892e+09,50809.468734,1,328,2025-03
4,AADI.JK,2025-04-30,5.236685e+13,NaN,1997-12-09,0.05425,0.385120,Consumer Non-Cyclicals,7.786892e+09,52366.847086,1,329,2025-04


In [ ]:
# Get the most recent fundamentals for each stock for each month
fund_latest = (
    fund.sort_values("Date")
    .groupby(["Instrument", "YearMonth"]) # group by month to get most recent fundamentals for each month for each stock
    .last() # get the most recent row for each group
    .reset_index()
)
fund_latest[fund_latest["BVPS"].notna()].head(5)

,Instrument,YearMonth,Date,Market_Cap,ROA,IPO_Date,DPS_Raw,BVPS,Sector,Shares_Outstanding,Market_Cap_Bil,Div_Payer,Months_Since_Listing
0,AADI.JK,2024-12,2024-12-30,6.599391e+13,NaN,1997-12-09,NaN,0.560686,Consumer Non-Cyclicals,7.786892e+09,65993.907666,0,325
1,AADI.JK,2025-01,2025-01-31,7.358613e+13,NaN,1997-12-09,NaN,0.560686,Consumer Non-Cyclicals,7.786892e+09,73586.127132,0,326
2,AADI.JK,2025-02,2025-02-28,5.236685e+13,NaN,1997-12-09,NaN,0.560686,Consumer Non-Cyclicals,7.786892e+09,52366.847086,0,327
3,AADI.JK,2025-03,2025-03-27,5.080947e+13,NaN,1997-12-09,0.05425,0.385120,Consumer Non-Cyclicals,7.786892e+09,50809.468734,1,328
4,AADI.JK,2025-04,2025-04-30,5.236685e+13,NaN,1997-12-09,0.05425,0.385120,Consumer Non-Cyclicals,7.786892e+09,52366.847086,1,329


In [10]:
# Lag by 1 month to match panel YearMonth 
fund_latest["Panel_YM"] = (
    pd.to_datetime(fund_latest["YearMonth"]) + pd.DateOffset(months=1)
).dt.to_period("M").astype(str)

merged = daily.merge(
    fund_latest[["Instrument", "Panel_YM", "Sector", "BVPS", "IPO_Date",
                 "Months_Since_Listing", "Market_Cap"]],
    left_on=["Instrument", "YearMonth"],
    right_on=["Instrument", "Panel_YM"],
    how="left",
)

N_total = len(merged)
N_stocks_total = merged["Instrument"].nunique()
print(f"Starting universe: {N_total:,} firm-days, {N_stocks_total} stocks")

Starting universe: 1,633,838 firm-days, 937 stocks


In [11]:
merged[merged["BVPS"].notna()].head(5)

,Date,Instrument,Price,Volume,Market_Return,Daily_Rf,Stock_Return,Excess_Return,Market_Excess,DayOfWeek,DayName,YearMonth,Period,Panel_YM,Sector,BVPS,IPO_Date,Months_Since_Listing,Market_Cap
37,2012-02-01,AALI.JK,19390.51785,2.828879e+06,0.005889,0.000153,-0.012210,-0.012363,0.005737,2,Wednesday,2012-02,Pre,2012-02,Consumer Non-Cyclicals,4363.662662,1997-12-09,170.0,3.243975e+13
38,2012-02-02,AALI.JK,20152.79865,3.154743e+06,0.013011,0.000153,0.038559,0.038406,0.012858,3,Thursday,2012-02,Pre,2012-02,Consumer Non-Cyclicals,4363.662662,1997-12-09,170.0,3.243975e+13
39,2012-02-03,AALI.JK,20486.29650,2.785850e+06,-0.000237,0.000153,0.016413,0.016260,-0.000390,4,Friday,2012-02,Pre,2012-02,Consumer Non-Cyclicals,4363.662662,1997-12-09,170.0,3.243975e+13
40,2012-02-06,AALI.JK,20915.07945,1.436741e+06,-0.010302,0.000153,0.020714,0.020561,-0.010455,0,Monday,2012-02,Pre,2012-02,Consumer Non-Cyclicals,4363.662662,1997-12-09,170.0,3.243975e+13
41,2012-02-07,AALI.JK,21153.29220,1.045809e+06,-0.004877,0.000153,0.011325,0.011172,-0.005029,1,Tuesday,2012-02,Pre,2012-02,Consumer Non-Cyclicals,4363.662662,1997-12-09,170.0,3.243975e+13


## 1. Price Filter (< IDR 1,000)

In [12]:
low_price = merged["Price"] < 1000
n_low = low_price.sum()
pct_low = n_low / N_total * 100

# Stocks that would be ENTIRELY removed (all obs below 1000)
stock_max_price = merged.groupby("Instrument")["Price"].max() # get max price for each stock
always_penny = stock_max_price[stock_max_price < 1000].index

# Stocks that lose SOME observations
stock_low_pct = (
    merged.groupby("Instrument")["Price"]
    .apply(lambda x: (x < 1000).mean() * 100)
    .sort_values(ascending=False)
)

print(f"Stocks always below 1,000: {len(always_penny)}")
print(f"Stocks with ANY obs below 1,000: {(stock_low_pct > 0).sum()}")

print("\nPrice distribution (all obs):")
for threshold in [50, 100, 200, 500, 1000, 2000, 5000]:
    n = (merged["Price"] < threshold).sum()
    print(f"  < IDR {threshold:>5,}: {n:>10,} obs ({n/N_total*100:5.1f}%)")

# After filter: how many stocks and obs remain?
after = merged[~low_price]
print(f"\nAfter filter: {len(after):,} firm-days ({len(after)/N_total*100:.1f}%), "
      f"{after['Instrument'].nunique()} stocks")

Stocks always below 1,000: 508
Stocks with ANY obs below 1,000: 873

Price distribution (all obs):
  < IDR    50:     47,118 obs (  2.9%)
  < IDR   100:    305,229 obs ( 18.7%)
  < IDR   200:    586,735 obs ( 35.9%)
  < IDR   500:    983,498 obs ( 60.2%)
  < IDR 1,000:  1,236,840 obs ( 75.7%)
  < IDR 2,000:  1,421,326 obs ( 87.0%)
  < IDR 5,000:  1,553,461 obs ( 95.1%)

After filter: 396,998 firm-days (24.3%), 430 stocks


In [13]:
merged[merged["Price"] < 1000].head(20)

,Date,Instrument,Price,Volume,Market_Return,Daily_Rf,Stock_Return,Excess_Return,Market_Excess,DayOfWeek,DayName,YearMonth,Period,Panel_YM,Sector,BVPS,IPO_Date,Months_Since_Listing,Market_Cap
3173,2012-01-02,ABBA.JK,96.139098,1.392004e+07,-0.003368,0.000153,NaN,NaN,-0.003521,0,Monday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3174,2012-01-03,ABBA.JK,94.550022,5.846165e+05,0.012715,0.000153,-0.016667,-0.016820,0.012562,1,Tuesday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3175,2012-01-04,ABBA.JK,93.755484,2.064093e+05,0.012759,0.000153,-0.008439,-0.008592,0.012606,2,Wednesday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3176,2012-01-05,ABBA.JK,94.550022,2.070386e+05,-0.000296,0.000153,0.008439,0.008286,-0.000449,3,Thursday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3177,2012-01-06,ABBA.JK,97.728174,1.432656e+07,-0.009478,0.000153,0.033061,0.032908,-0.009631,4,Friday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3178,2012-01-09,ABBA.JK,97.728174,1.913061e+05,0.005067,0.000153,0.000000,-0.000153,0.004914,0,Monday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3179,2012-01-10,ABBA.JK,93.755484,7.771812e+05,0.012716,0.000153,-0.041500,-0.041653,0.012563,1,Tuesday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3180,2012-01-11,ABBA.JK,93.755484,2.038921e+05,-0.007441,0.000153,0.000000,-0.000153,-0.007594,2,Wednesday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3181,2012-01-12,ABBA.JK,94.550022,3.775779e+04,-0.000037,0.000153,0.008439,0.008286,-0.000189,3,Thursday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN
3182,2012-01-13,ABBA.JK,94.550022,4.530935e+04,0.006585,0.000153,0.000000,-0.000153,0.006432,4,Friday,2012-01,Pre,NaN,NaN,NaN,NaT,NaN,NaN


## 2. Seasoning Requirement (< 12 months post-IPO)

In [14]:
has_age = merged["Months_Since_Listing"].notna()
unseasoned = merged["Months_Since_Listing"] < 12

n_unseasoned = unseasoned.sum()
pct_unseasoned = n_unseasoned / N_total * 100

# Which stocks are affected?
unseasoned_stocks = merged[unseasoned]["Instrument"].unique()

print(f"Firm-days with < 12 months listing history: {n_unseasoned:,} / {N_total:,} ({pct_unseasoned:.1f}%)")
print(f"Stocks with any unseasoned obs: {len(unseasoned_stocks)}")
print(f"Stocks missing age data entirely: {(~has_age).groupby(merged['Instrument']).all().sum()}")

# Age distribution
age = merged[has_age]["Months_Since_Listing"]
print(f"\nAge distribution (months since listing):")
print(f"  Mean:   {age.mean():.0f}")
print(f"  Median: {age.median():.0f}")
print(f"  Min:    {age.min():.0f}")
print(f"  P5:     {age.quantile(0.05):.0f}")
print(f"  P10:    {age.quantile(0.10):.0f}")

# Year breakdown of unseasoned obs
if n_unseasoned > 0:
    print(f"\nUnseasoned obs by year:")
    print(merged[unseasoned].groupby(merged[unseasoned]["Date"].dt.year).size())

Firm-days with < 12 months listing history: 119,264 / 1,633,838 (7.3%)
Stocks with any unseasoned obs: 498
Stocks missing age data entirely: 23

Age distribution (months since listing):
  Mean:   162
  Median: 137
  Min:    -140
  P5:     7
  P10:    16

Unseasoned obs by year:
Date
2012     6788
2013     7663
2014     7137
2015     5171
2016     5434
2017     6002
2018    11533
2019    13515
2020    14311
2021     9584
2022    11148
2023    13588
2024     7390
dtype: int64


## 3. Zero-Volume Filter

In [15]:
zero_vol = merged["Volume"] == 0
n_zero = zero_vol.sum()
pct_zero = n_zero / N_total * 100

print(f"Firm-days with zero volume: {n_zero:,} / {N_total:,} ({pct_zero:.1f}%)")

# Per-stock zero-volume rates
stock_zero_pct = (
    merged.groupby("Instrument")["Volume"]
    .apply(lambda x: (x == 0).mean() * 100)
    .sort_values(ascending=False)
)

print(f"\nStocks by zero-volume rate:")
for threshold in [0, 10, 25, 50, 75, 90]:
    n = (stock_zero_pct > threshold).sum()
    print(f"  > {threshold}% zero-vol days: {n} stocks")

# Are zero-volume days concentrated in small stocks?
has_mcap = merged["Market_Cap"].notna()
if has_mcap.any():
    merged["MCap_Q"] = pd.Series(dtype="object")
    merged.loc[has_mcap, "MCap_Q"] = pd.qcut(
        merged.loc[has_mcap, "Market_Cap"].rank(method="first"),
        5, labels=["Q1(small)", "Q2", "Q3", "Q4", "Q5(large)"]
    )
    print(f"\nZero-volume rate by market cap quintile:")
    for q in ["Q1(small)", "Q2", "Q3", "Q4", "Q5(large)"]:
        sub = merged[merged["MCap_Q"] == q]
        rate = (sub["Volume"] == 0).mean() * 100
        print(f"  {q}: {rate:.1f}%")

# Zero-vol days by day-of-week
print(f"\nZero-volume rate by day of week:")
for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]:
    sub = merged[merged["DayName"] == day]
    rate = (sub["Volume"] == 0).mean() * 100
    print(f"  {day}: {rate:.1f}%")

Firm-days with zero volume: 0 / 1,633,838 (0.0%)

Stocks by zero-volume rate:
  > 0% zero-vol days: 0 stocks
  > 10% zero-vol days: 0 stocks
  > 25% zero-vol days: 0 stocks
  > 50% zero-vol days: 0 stocks
  > 75% zero-vol days: 0 stocks
  > 90% zero-vol days: 0 stocks

Zero-volume rate by market cap quintile:
  Q1(small): 0.0%
  Q2: 0.0%
  Q3: 0.0%
  Q4: 0.0%
  Q5(large): 0.0%

Zero-volume rate by day of week:
  Monday: 0.0%
  Tuesday: 0.0%
  Wednesday: 0.0%
  Thursday: 0.0%
  Friday: 0.0%


## 4. Financial Sector Exclusion

In [16]:
has_sector = merged["Sector"].notna()
is_financial = merged["Sector"].str.contains("Financ", case=False, na=False)

n_fin = is_financial.sum()
pct_fin = n_fin / N_total * 100
fin_stocks = merged[is_financial]["Instrument"].nunique()

print(f"Financial sector firm-days: {n_fin:,} / {N_total:,} ({pct_fin:.1f}%)")
print(f"Financial sector stocks: {fin_stocks}")
print(f"Stocks missing sector data: {(~has_sector).groupby(merged['Instrument']).all().sum()}")

# Sector breakdown
print(f"\nFull sector breakdown (firm-days):")
sector_counts = (
    merged[has_sector]
    .groupby("Sector")
    .agg(firm_days=("Date", "count"), stocks=("Instrument", "nunique"))
    .sort_values("firm_days", ascending=False)
)
sector_counts["pct"] = sector_counts["firm_days"] / sector_counts["firm_days"].sum() * 100
print(sector_counts.to_string())

# Financial sub-sectors if available
fin_sectors = merged[is_financial]["Sector"].value_counts()
if len(fin_sectors) > 1:
    print(f"\nFinancial sub-categories:")
    print(fin_sectors)

Financial sector firm-days: 229,660 / 1,633,838 (14.1%)
Financial sector stocks: 104
Stocks missing sector data: 23

Full sector breakdown (firm-days):
                        firm_days  stocks        pct
Sector                                              
Consumer Cyclicals         267528     163  16.631149
Industrials                244264     149  15.184919
Financials                 229660     104  14.277047
Consumer Non-Cyclicals     217040     123  13.492511
Basic Materials            212389     114  13.203377
Real Estate                157794      87   9.809424
Energy                     117657      63   7.314267
Technology                  90286      63   5.612721
Healthcare                  54442      35   3.384442
Utilities                   17536      13   1.090143


## 5. Combined Filter Summary

In [17]:
# Apply filters sequentially to show cumulative attrition
print("CUMULATIVE FILTER IMPACT")
print("=" * 70)

df = merged.copy()
print(f"{'Starting universe':<40s} {len(df):>10,} firm-days  {df['Instrument'].nunique():>5} stocks")

# 1. Price filter
df = df[df["Price"] >= 1000]
print(f"{'After Price >= IDR 1,000':<40s} {len(df):>10,} firm-days  {df['Instrument'].nunique():>5} stocks")

# 2. Seasoning
df = df[(df["Months_Since_Listing"].isna()) | (df["Months_Since_Listing"] >= 12)]
print(f"{'After seasoning >= 12 months':<40s} {len(df):>10,} firm-days  {df['Instrument'].nunique():>5} stocks")

# 3. Zero volume
df = df[df["Volume"] > 0]
print(f"{'After zero-volume drop':<40s} {len(df):>10,} firm-days  {df['Instrument'].nunique():>5} stocks")

# 4. Financial sector
df = df[~df["Sector"].str.contains("Financ", case=False, na=False)]
print(f"{'After financial sector exclusion':<40s} {len(df):>10,} firm-days  {df['Instrument'].nunique():>5} stocks")

print(f"\n{'FINAL SAMPLE':<40s} {len(df):>10,} firm-days  {df['Instrument'].nunique():>5} stocks")
print(f"{'Attrition':<40s} {N_total - len(df):>10,} firm-days  "
      f"{N_stocks_total - df['Instrument'].nunique():>5} stocks")
print(f"{'Retention rate':<40s} {len(df)/N_total*100:>9.1f}%")

# Note: Negative BVPS is NOT a sample-wide filter
neg_bvps_in_final = (df["BVPS"] < 0).sum()
print(f"\nNote: {neg_bvps_in_final:,} firm-days in final sample have BVPS < 0")
print(f"  -> excluded from B/M sorts only, retained in all other anomaly sorts")

CUMULATIVE FILTER IMPACT
Starting universe                         1,633,838 firm-days    937 stocks
After Price >= IDR 1,000                    396,997 firm-days    429 stocks
After seasoning >= 12 months                376,373 firm-days    413 stocks
After zero-volume drop                      376,373 firm-days    413 stocks
After financial sector exclusion            315,923 firm-days    376 stocks

FINAL SAMPLE                                315,923 firm-days    376 stocks
Attrition                                 1,317,915 firm-days    561 stocks
Retention rate                                19.3%

Note: 1,759 firm-days in final sample have BVPS < 0
  -> excluded from B/M sorts only, retained in all other anomaly sorts
